In [2]:
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler
from pathlib import Path

# 1. Setup Paths and Load Data
PROCESSED_DIR = Path("../data/processed")
edges_path = PROCESSED_DIR / "bis_cbs_quarterly_2000Q1_2026Q1_graph_edges.csv"

print(f"Loading data from: {edges_path}")
df = pd.read_csv(edges_path)

# 2. Chronological Splitting (as defined in the preprocessing report)
train_mask = (df['year'] >= 2000) & (df['year'] <= 2018)
val_mask = (df['year'] >= 2019) & (df['year'] <= 2021)
test_mask = (df['year'] >= 2022) & (df['year'] <= 2026)

df_train = df[train_mask].copy()
df_val = df[val_mask].copy()
df_test = df[test_mask].copy()

print(f"Training edges: {len(df_train):,}")
print(f"Validation edges: {len(df_val):,}")
print(f"Testing edges: {len(df_test):,}")

# 3. Prevent Data Leakage: Fit Scaler ONLY on Training Data
scaler = StandardScaler()
df_train['scaled_log_claim'] = scaler.fit_transform(df_train[['log_claim_value']])
df_val['scaled_log_claim'] = scaler.transform(df_val[['log_claim_value']])
df_test['scaled_log_claim'] = scaler.transform(df_test[['log_claim_value']])

# Combine back into a single dataframe for easy snapshot iteration
df_scaled = pd.concat([df_train, df_val, df_test]).sort_values('period_index')

# 4. Construct PyTorch Geometric Data Objects
graph_snapshots = []
periods = df_scaled['period'].unique()

for period in periods:
    df_period = df_scaled[df_scaled['period'] == period]
    
    # Extract edges (source -> target)
    source_nodes = torch.tensor(df_period['source_node_id'].values, dtype=torch.long)
    target_nodes = torch.tensor(df_period['target_node_id'].values, dtype=torch.long)
    edge_index = torch.stack([source_nodes, target_nodes], dim=0)
    
    # Extract edge weights
    edge_attr = torch.tensor(df_period['scaled_log_claim'].values, dtype=torch.float).view(-1, 1)
    
    # Create PyG Data object
    graph = Data(edge_index=edge_index, edge_attr=edge_attr)
    graph.period = period  # Attach metadata for tracking
    
    graph_snapshots.append(graph)

print(f"\nSuccessfully created {len(graph_snapshots)} sequential PyG graph snapshots.")
print(f"Sample Graph (First Quarter): {graph_snapshots[0]}")

# 5. Save the Tensors for Deep Learning Models
torch.save(graph_snapshots, PROCESSED_DIR / "temporal_graph_sequence.pt")
print("Saved temporal graph sequence to data/processed/temporal_graph_sequence.pt")

Loading data from: ..\data\processed\bis_cbs_quarterly_2000Q1_2026Q1_graph_edges.csv
Training edges: 145,145
Validation edges: 30,765
Testing edges: 43,089

Successfully created 105 sequential PyG graph snapshots.
Sample Graph (First Quarter): Data(edge_index=[2, 1270], edge_attr=[1270, 1], period='2000-Q1')
Saved temporal graph sequence to data/processed/temporal_graph_sequence.pt
